# Classical baselines on the validation split

`passage-eval` ships three classical planners that need no training: exact A\*, a greedy walk toward
the goal, and a random walk closed by A\*. This notebook scores them on 64 validation tasks at
256×256, for each of the three objectives with the obstacles blocked, and gives a distribution-free
95 % interval for each median regret.

These are reference points for reading the metrics, computed on the validation split. They are not
benchmark results: a benchmark result freezes its settings on the calibration split and then makes one
pass on the test split.

**Run it yourself.** From a checkout of the [code repository](https://github.com/ThalesGroup/passage), start Jupyter with `uv run --frozen jupyter lab`, open this notebook and point `dataset_path` to a local copy of [the dataset](https://huggingface.co/datasets/thalesgroup/passage). With `dataset_path = None`, rows are read from the Hub at the pinned `dataset_revision` instead. It runs about 600 planner calls; one to two minutes.

In [ ]:
resolution = 256
split = "validation"
condition = "obstacles"
tasks = 64  # validation tasks per objective
seed = 0  # seed of the random-walk planner
output_dir = "notebook-artifacts/notebooks/passage-eval/02_score_released_baselines"
dataset_path = "dataset"  # local copy of the dataset; None reads the Hub at dataset_revision
dataset_repo = "thalesgroup/passage"
dataset_revision = "2259dd94e2de28ab4a56c270247a7e347ec8f133"  # verified PASSAGE v1.0 upload
project_path = None  # checkout root; None searches upwards from the working directory

In [ ]:
from pathlib import Path
import json  # noqa: F401
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np  # noqa: F401
import pandas as pd
from IPython.display import Markdown, display

# Locate the code checkout (pyproject.toml next to src/passage) without changing directory.
project_root = Path(project_path).resolve() if project_path else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src/passage").is_dir()), Path.cwd())
for extra in (project_root / "src", project_root / "scripts"):
    if extra.is_dir() and str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from passage.notebook_reports import make_source, resolve_path, save_figure, write_report  # noqa: E402, F401

source = make_source(dataset_path, dataset_repo=dataset_repo, dataset_revision=dataset_revision,
                     project_root=project_root)
output_path = resolve_path(output_dir, project_root)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)


def relative(path):
    """Show saved files relative to the checkout, never as machine paths."""
    path = Path(path)
    try:
        return path.resolve().relative_to(project_root.resolve()).as_posix()
    except ValueError:
        return path.name


def show(frame, caption=None, floatfmt="{:,.3f}"):
    """Display a table with readable numbers and an optional caption."""
    if caption:
        display(Markdown(f"*{caption}*"))
    floats = list(frame.select_dtypes("float").columns)
    ints = list(frame.select_dtypes("integer").columns)
    others = [c for c in frame.columns if c not in floats and c not in ints]
    styler = frame.style.hide(axis="index")
    for columns, fmt in ((floats, floatfmt), (ints, "{:,}"), (others, None)):
        if columns:
            styler = styler.format(fmt, subset=columns, na_rep="—")
    display(styler)


# One chart style for every PASSAGE notebook: recessive axes, light grid, left-aligned titles.
INK, MUTED, GRID = "#243537", "#5c6765", "#e3e6e3"
OBJECTIVE_COLORS = {"elevation": "#1767ac", "slope_uphill": "#d56418", "energy": "#a32b8e"}
OBJECTIVE_LABELS = {"elevation": "Elevation", "slope_uphill": "Slope (uphill)", "energy": "Energy"}
SPLIT_COLORS = {"train": "#1f8a70", "calibration": "#b07d1a", "validation": "#3a62b5", "test": "#c0392b"}
REGIME_COLORS = {"free": "#238b45", "obstacles": "#1767ac"}
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "#8a9290", "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.titlelocation": "left",
    "axes.titlesize": 10.5, "axes.titleweight": "semibold", "axes.titlecolor": INK, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False, "lines.linewidth": 1.6,
    "figure.dpi": 110, "savefig.dpi": 140, "pdf.fonttype": 42, "ps.fonttype": 42,
})


def figure_title(fig, title, subtitle=None):
    """Left-aligned title and subtitle in a band reserved at the top of the figure."""
    height = fig.get_figheight()
    fig.suptitle(title, x=0.01, y=1 - 0.1 / height, ha="left", va="top", fontsize=12, fontweight="semibold", color=INK)
    if subtitle:
        fig.text(0.01, 1 - 0.4 / height, subtitle, ha="left", va="top", fontsize=8.5, color=MUTED)
    fig.subplots_adjust(top=min(fig.subplotpars.top, 1 - 0.95 / height))


from passage_eval.baselines.classical import AStarPlanner, GreedyToGoalPlanner, RandomPolicyPlanner  # noqa: E402, F401
from passage_eval.io.dataset import scene_from_row  # noqa: E402
from passage_eval.protocol.planner import Planner, PlannerOutput, evaluate_planner  # noqa: E402, F401

METRICS = ["cost_regret_median", "cost_regret_p90", "path_assd", "path_hausdorff_p95",
           "success_rate", "obstacle_violation_rate"]
PLANNER_COLORS = {"A*": "#238b45", "Greedy": "#1767ac", "Random walk": "#b8862b", "Straight line": "#1767ac"}


SOLVER_OF = {"elevation": "astar_elevation", "slope_uphill": "astar_slope", "energy": "astar_energy"}


def load_tasks(resolution, objective, condition, limit):
    """First `limit` scored tasks of the split; a task without a reference route would be skipped, and in v1.0 every task has one."""
    columns = ["id", "tensor", "metadata"] + [f"path_astar_{s}_{condition}" for s in ("elevation", "slope", "energy")]
    tasks = []
    for row in source.iter_rows(resolution, split, columns=columns, samples=False):
        if row["metadata"]["paths"][SOLVER_OF[objective]][condition]["status"] != "ok":
            continue
        tasks.append(scene_from_row(row, resolution=resolution, cost_model=objective, split=split, task_variant=condition))
        if len(tasks) == limit:
            return tasks
    return tasks

## 1. Scores per objective

Each objective gets its own tasks: the same scenes, priced and labelled under that objective.

In [ ]:
from scipy.stats import binom

OBJECTIVES = ("elevation", "slope_uphill", "energy")


def median_interval(values, level=0.95):
    """Distribution-free interval for the median, from the order statistics of the tasks."""
    ordered = np.sort(np.asarray(values, dtype=float))
    rank = int(binom.ppf((1 - level) / 2, ordered.size, 0.5))  # 1-based rank of the lower bound
    if rank < 1:
        return ordered[0], ordered[-1]
    return ordered[rank - 1], ordered[ordered.size - rank]


rows, per_task = [], {}
for objective in OBJECTIVES:
    objective_tasks = load_tasks(resolution, objective, condition, tasks)
    planners = {"A*": AStarPlanner(), "Greedy": GreedyToGoalPlanner(), "Random walk": RandomPolicyPlanner(seed=seed)}
    for name, planner in planners.items():
        metrics, arrays = evaluate_planner(planner, objective_tasks)
        regret = np.asarray(arrays["regret"], dtype=float)
        low, high = median_interval(regret)
        rows.append({"Objective": OBJECTIVE_LABELS[objective], "Planner": name,
                     **{m: metrics[m] for m in METRICS}, "Median regret, 95 % interval": f"[{low:.4f}, {high:.4f}]"})
        per_task[(objective, name)] = {"regret": regret, "valid_path": arrays["valid_path"]}
scores = pd.DataFrame(rows)
show(scores, f"Table 1. Six primary metrics on {tasks} validation tasks per objective, {resolution}×{resolution}, {condition}.", "{:,.4f}")

The interval comes from the order statistics of the tasks: with 64 tasks, the 24th and 41st smallest
regrets bound the median with at least 95 % confidence, whatever the distribution. It treats tasks as
independent, so it does not cover the variation between geographic blocks, which the paper's protocol
handles separately.

## 2. Regret per task

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
fig.subplots_adjust(top=0.76, wspace=0.08)
names = ["A*", "Greedy", "Random walk"]
for ax, objective in zip(axes, OBJECTIVES):
    for index, name in enumerate(names):
        regret = per_task[(objective, name)]["regret"]
        failed = ~np.asarray(per_task[(objective, name)]["valid_path"], dtype=bool)
        shown = np.clip(100 * regret[~failed], 1e-3, None)
        jitter = np.linspace(-0.15, 0.15, num=shown.size)  # deterministic spread of the points
        ax.scatter(shown, index + jitter, s=12, color=PLANNER_COLORS[name], alpha=0.7, linewidths=0)
        if failed.any():
            ax.annotate(f"{int(failed.sum())} failed", (1.0, index), xycoords=("axes fraction", "data"),
                        xytext=(-2, 9), textcoords="offset points", ha="right", fontsize=7.5, color=MUTED)
    ax.set_xscale("log")
    ax.set_title(OBJECTIVE_LABELS[objective], color=OBJECTIVE_COLORS[objective])
    ax.set_xlabel("Cost regret (%, log)")
    ax.grid(True, axis="x")
axes[0].set_yticks(range(len(names)), names)
axes[0].invert_yaxis()
figure_title(fig, "Regret of the classical baselines per task",
             f"{tasks} {split} tasks per objective · {resolution}×{resolution} · {condition} · zero regret drawn at 0.001 %")
figures = save_figure(fig, output_path, f"02_score_released_baselines-{resolution}")
plt.show()

*Figure 1. Cost regret of each task for the three planners, one panel per objective, on a log scale.
Failed tasks (regret 999,999) are counted on the right of each row.*

## Summary

In [ ]:
g = scores[scores.Planner == "Greedy"].set_index("Objective")
r = scores[scores.Planner == "Random walk"].set_index("Objective")
display(Markdown("\n".join([
    "- A\\* matches the labels on every objective: zero regret and full success, as it must.",
    "- Greedy connects " + ", ".join(f"{100 * v:.0f} % ({k.lower()})" for k, v in g.success_rate.items())
    + " of the tasks; its failures dominate its 90th-percentile regret.",
    "- The random walk always connects; its median regret is "
    + ", ".join(f"{100 * v:.2f} % ({k.lower()})" for k, v in r.cost_regret_median.items()) + ".",
])))
report = {"resolution": resolution, "split": split, "condition": condition, "tasks": tasks,
          "scores": scores.to_dict("records"), "figures": [relative(f) for f in figures]}

In [ ]:
report_file = write_report(report, output_path, f"02_score_released_baselines-{resolution}.json")
display(Markdown(f"The numbers of this run are saved in `{relative(report_file)}`, next to the figures."))

---
*MIT License · Copyright (c) 2026 THALES. The full licence text is in the `LICENSE` file of the repository.*